In [163]:
"""
LICENSE MIT
2021
Guillaume Rozier
Website : http://www.covidtracker.fr
Mail : guillaume.rozier@telecomnancy.net

README:
This file contains scripts that download data from data.gouv.fr and then process it to build many graphes.
I'm currently cleaning the code, please ask me if something is not clear enough.

The charts are exported to 'charts/images/france'.
Data is download to/imported from 'data/france'.
Requirements: please see the imports below (use pip3 to install them).

"""

"\nLICENSE MIT\n2021\nGuillaume Rozier\nWebsite : http://www.covidtracker.fr\nMail : guillaume.rozier@telecomnancy.net\n\nREADME:\nThis file contains scripts that download data from data.gouv.fr and then process it to build many graphes.\nI'm currently cleaning the code, please ask me if something is not clear enough.\n\nThe charts are exported to 'charts/images/france'.\nData is download to/imported from 'data/france'.\nRequirements: please see the imports below (use pip3 to install them).\n\n"

In [164]:
import pandas as pd
import json
import france_data_management as data
import math

show_charts = False
PATH_STATS = "../../data/france/stats/"

In [165]:
df, df_confirmed, dates, df_new, df_tests, df_deconf, df_sursaud, df_incid, df_tests_viros = data.import_data()









  0%|          | 0/8 [00:00<?, ?it/s]







 38%|███▊      | 3/8 [00:02<00:04,  1.11it/s]







 75%|███████▌  | 6/8 [00:05<00:01,  1.06it/s]







10it [00:06,  1.48it/s]                      







15it [00:06,  2.04it/s]







21it [00:19,  1.02it/s]







28it [02:57,  7.48s/it]







36it [02:57,  5.24s/it]

In [178]:
df_incid_fra_clage = data.import_data_tests_sexe()
df_incid_fra = df_incid_fra_clage[df_incid_fra_clage["cl_age90"]==0]
df_france = df.groupby(["jour"]).sum().reset_index()
df_incid = df_incid[df_incid.cl_age90 == 0]

df_sursaud_france = df_sursaud.groupby(["date_de_passage"]).sum().reset_index()
df_sursaud_regions = df_sursaud.groupby(["date_de_passage", "regionName"]).sum().reset_index()

df_new_france = df_new.groupby(["jour"]).sum().reset_index()
df_new_regions = df_new.groupby(["jour", "regionName"]).sum().reset_index()

In [167]:
departements = list(dict.fromkeys(list(df_incid['dep'].values))) 
regions = list(dict.fromkeys(list(df_incid['regionName'].dropna().values))) 
clage_list = list(dict.fromkeys(list(df_incid_fra_clage['cl_age90'].dropna().values))) 

df_regions = df.groupby(["jour", "regionName"]).sum().reset_index()
df_incid_regions = df_incid.groupby(["jour", "regionName"]).sum().reset_index()

In [205]:
def generate_data(data_incid, data_hosp, data_sursaud, data_new):## Incidence
        
    dict_data = {}

    taux_incidence = data_incid["P"].rolling(window=7).sum().fillna(0) * 100000 / data_incid["pop"].values[0]
    dict_data["incidence"] = {"jour": list(data_incid.jour), "valeur": list(round(taux_incidence,2))}
    
    taux_positivite = (data_incid["P"].rolling(window=7).sum() * 100 / data_incid["T"].rolling(window=7).sum()).fillna(0)
    dict_data["taux_positivite"] = {"jour": list(data_incid.jour), "valeur": list(round(taux_positivite,2))}

    cas = data_incid["P"].rolling(window=7).mean().fillna(0)
    dict_data["cas"] = {"jour": list(data_incid.jour), "valeur": list(round(cas,2))}
    
    tests = data_incid["T"].rolling(window=7).mean().fillna(0)
    dict_data["tests"] = {"jour": list(data_incid.jour), "valeur": list(round(tests,2))}

    hospitalisations = data_hosp.hosp.fillna(0)
    dict_data["hospitalisations"] = {"jour": list(data_hosp.jour), "valeur": list(hospitalisations)}

    reanimations = data_hosp.rea.fillna(0)
    dict_data["reanimations"] = {"jour": list(data_hosp.jour), "valeur": list(reanimations)}
    
    incid_hospitalisations = data_new.incid_hosp.rolling(window=7).mean().fillna(0)
    dict_data["incid_hospitalisations"] = {"jour": list(data_new.jour), "valeur": list(round(incid_hospitalisations, 2))}
    
    incid_reanimations = data_new.incid_rea.rolling(window=7).mean().fillna(0)

    try:
        if(data_new.regionName ==regions[0]):
            print(data_new.jour)
    except:
        pass
    dict_data["incid_reanimations"] = {"jour": list(data_new.jour), "valeur": list(round(incid_reanimations,2))}
    
    nbre_acte_corona = data_sursaud.nbre_acte_corona.rolling(window=7).mean().fillna(0)
    dict_data["nbre_acte_corona"] = {"jour": list(data_sursaud.date_de_passage), "valeur": list(round(nbre_acte_corona, 2))}
    
    nbre_pass_corona = data_sursaud.nbre_pass_corona.rolling(window=7).mean().fillna(0)
    dict_data["nbre_pass_corona"] = {"jour": list(data_sursaud.date_de_passage), "valeur": list(round(nbre_pass_corona, 2))}

    deces_hospitaliers = data_hosp.dc.diff().rolling(window=7).mean().fillna(0)
    dict_data["deces_hospitaliers"] = {"jour": list(data_hosp.jour), "valeur": list(round(deces_hospitaliers,2))}
    
    population = data_incid["pop"].values[0]
    dict_data["population"] = population
    
    return dict_data
 

In [180]:
def generate_data_age(data_incid, data_hosp, clage_list):## Incidence
    dict_data = {}
    
    for clage in clage_list:
        dict_data[clage] = {}
        taux_incidence = data_incid["P"].rolling(window=7).sum().fillna(0) * 100000 / data_incid["pop"].values[0]
        dict_data[clage]["incidence"] = {"jour": list(data_incid.jour), "valeur": list(round(taux_incidence,2))}

        taux_positivite = (data_incid["P"].rolling(window=7).sum() * 100 / data_incid["T"].rolling(window=7).sum()).fillna(0)
        dict_data[clage]["taux_positivite"] = {"jour": list(data_incid.jour), "valeur": list(round(taux_positivite,2))}

        cas = data_incid["P"].rolling(window=7).mean().fillna(0)
        dict_data[clage]["cas"] = {"jour": list(data_incid.jour), "valeur": list(round(cas,2))}

        tests = data_incid["T"].rolling(window=7).mean().fillna(0)
        dict_data[clage]["tests"] = {"jour": list(data_incid.jour), "valeur": list(round(tests,2))}

        hospitalisations = data_hosp.hosp.fillna(0)
        dict_data[clage]["hospitalisations"] = {"jour": list(data_hosp.jour), "valeur": list(hospitalisations)}

        reanimations = data_hosp.rea.fillna(0)
        dict_data[clage]["reanimations"] = {"jour": list(data_hosp.jour), "valeur": list(reanimations)}

        deces_hospitaliers = data_hosp.dc.diff().rolling(window=7).mean().fillna(0)
        dict_data[clage]["deces_hospitaliers"] = {"jour": list(data_hosp.jour), "valeur": list(round(deces_hospitaliers,2))}

        population = data_incid["pop"].values[0]
        dict_data["population"] = population
    
    return dict_data
 

In [181]:
def export_data(data, suffix=""):
    with open(PATH_STATS + 'dataexplorer{}.json'.format(suffix), 'w') as outfile:
        json.dump(data, outfile)

In [182]:
def dataexplorer():
    dict_data = {}
    dict_data["regions"] = sorted(regions)
    dict_data["departements"] = departements
    dict_data["france"] = generate_data(df_incid_fra, df_france, df_sursaud_france, df_new_france)
    
    noms_departements={}
    
    for reg in regions:
        dict_data[reg] = generate_data(df_incid_regions[df_incid_regions.regionName==reg], \
                                       df_regions[df_regions.regionName==reg],\
                                       df_sursaud_regions[df_sursaud_regions.regionName==reg],
                                       df_new_regions[df_new_regions.regionName==reg])
    
    for dep in departements:
        df_incid_dep = df_incid[df_incid.dep==dep]
        dict_data[dep] = generate_data(df_incid_dep, df[df.dep==dep], df_sursaud[df_sursaud.dep==dep], df_new[df_new.dep==dep])
        
        noms_departements[dep] = df_incid_dep["departmentName"].values[0]
    dict_data["departements_noms"] = noms_departements
    
    export_data(dict_data)

In [193]:
df_new_regions[df_new_regions.regionName==regions[0]]

,jour,regionName,incid_hosp,incid_rea,incid_dc,incid_rad,regionCode,regionPopulation,code,departmentPopulation,incid_hosp_nonrea
0,2020-03-19,Auvergne-Rhône-Alpes,337,44,21,76,1008.0,96324000,1008,8069287,293
18,2020-03-20,Auvergne-Rhône-Alpes,98,16,5,34,1008.0,96324000,1008,8069287,82
36,2020-03-21,Auvergne-Rhône-Alpes,152,15,7,24,1008.0,96324000,1008,8069287,137
54,2020-03-22,Auvergne-Rhône-Alpes,133,25,12,20,1008.0,96324000,1008,8069287,108
72,2020-03-23,Auvergne-Rhône-Alpes,230,45,18,34,1008.0,96324000,1008,8069287,185
...,...,...,...,...,...,...,...,...,...,...,...
5940,2021-02-12,Auvergne-Rhône-Alpes,207,28,39,198,1008.0,96324000,1008,8069287,179
5958,2021-02-13,Auvergne-Rhône-Alpes,93,23,20,127,1008.0,96324000,1008,8069287,70
5976,2021-02-14,Auvergne-Rhône-Alpes,83,10,20,38,1008.0,96324000,1008,8069287,73
5994,2021-02-15,Auvergne-Rhône-Alpes,230,62,38,145,1008.0,96324000,1008,8069287,168


In [172]:
#generate_data(df_incid_fra_clage, df_france, df_sursaud_france, clage_list)

In [ ]:
dataexplorer()

           jour  incid_hosp  incid_rea  incid_dc  incid_rad  regionCode  \
0    2020-03-19        2229        438       155        519      5318.0   
1    2020-03-20        1256        242        83        317      5318.0   
2    2020-03-21        1540        298       115        368      5318.0   
3    2020-03-22        1534        309       124        315      5318.0   
4    2020-03-23        2053        448       189        377      5318.0   
..          ...         ...        ...       ...        ...         ...   
330  2021-02-12        1405        264       320       1561      5318.0   
331  2021-02-13        1085        191       199       1089      5318.0   
332  2021-02-14         651        113       167        280      5318.0   
333  2021-02-15        1431        306       413        934      5318.0   
334  2021-02-16        1857        323       352       1740      5318.0   

     regionPopulation  code  departmentPopulation  incid_hosp_nonrea  
0           546232349  5318 

      dep        jour  incid_hosp  incid_rea  incid_dc  incid_rad  \
2345   08  2020-03-19           1          1         0          0   
2346   08  2020-03-20           0          0         0          0   
2347   08  2020-03-21           2          0         0          0   
2348   08  2020-03-22           1          0         0          0   
2349   08  2020-03-23           1          1         0          0   
...    ..         ...         ...        ...       ...        ...   
29810  88  2021-02-12           5          0         2          5   
29811  88  2021-02-13          13          2         3          9   
29812  88  2021-02-14           3          0         2          0   
29813  88  2021-02-15          10          2         2          3   
29814  88  2021-02-16           6          0         6          8   

      departmentCode departmentName  regionCode regionName  regionPopulation  \
2345              08       Ardennes        44.0  Grand Est           5518000   
2346       

      dep        jour  incid_hosp  incid_rea  incid_dc  incid_rad  \
5695   18  2020-03-19           0          0         0          0   
5696   18  2020-03-20           0          0         0          0   
5697   18  2020-03-21           0          0         0          0   
5698   18  2020-03-22           0          0         0          0   
5699   18  2020-03-23           0          0         0          0   
...    ..         ...         ...        ...       ...        ...   
15405  45  2021-02-12          15          3         7         23   
15406  45  2021-02-13           9          2         1          8   
15407  45  2021-02-14           5          1         2          3   
15408  45  2021-02-15           7          5         7         20   
15409  45  2021-02-16          16          4         7         16   

      departmentCode departmentName  regionCode regionName  regionPopulation  \
5695              18           Cher        24.0     Centre           2567000   
5696       

       dep        jour  incid_hosp  incid_rea  incid_dc  incid_rad  \
32160  971  2020-03-19           0          0         0          0   
32161  971  2020-03-20           0          0         0          0   
32162  971  2020-03-21           0          0         0          0   
32163  971  2020-03-22           6          5         0          0   
32164  971  2020-03-23           0          0         0          0   
...    ...         ...         ...        ...       ...        ...   
32490  971  2021-02-12           1          0         0          3   
32491  971  2021-02-13           1          1         2          8   
32492  971  2021-02-14           0          0         0          0   
32493  971  2021-02-15           0          0         0          0   
32494  971  2021-02-16           3          0         1          2   

      departmentCode departmentName  regionCode  regionName  regionPopulation  \
32160            971     Guadeloupe         1.0  Guadeloupe            395700 

    dep        jour  incid_hosp  incid_rea  incid_dc  incid_rad  \
0    01  2020-03-19           1          0         0          0   
1    01  2020-03-20           0          0         0          1   
2    01  2020-03-21           3          0         0          0   
3    01  2020-03-22           3          1         0          1   
4    01  2020-03-23          14          1         0          5   
..   ..         ...         ...        ...       ...        ...   
330  01  2021-02-12          12          2         3         20   
331  01  2021-02-13           2          1         0          3   
332  01  2021-02-14           7          0         0          2   
333  01  2021-02-15           8          0         2         12   
334  01  2021-02-16          27          4         2          8   

    departmentCode departmentName  regionCode            regionName  \
0               01            Ain        84.0  Auvergne-Rhône-Alpes   
1               01            Ain        84.0  Auverg

...

      dep        jour  incid_hosp  incid_rea  incid_dc  incid_rad  \
24455  73  2020-03-19           9          2         0          4   
24456  73  2020-03-20           5          0         0          1   
24457  73  2020-03-21           2          0         0          0   
24458  73  2020-03-22           8          1         0          0   
24459  73  2020-03-23           6          2         1          5   
...    ..         ...         ...        ...       ...        ...   
24785  73  2021-02-12           6          0         2         12   
24786  73  2021-02-13           3          3         1          7   
24787  73  2021-02-14           1          0         1          1   
24788  73  2021-02-15          14          3         4          7   
24789  73  2021-02-16           7          0         3          6   

      departmentCode departmentName  regionCode            regionName  \
24455             73         Savoie        84.0  Auvergne-Rhône-Alpes   
24456             73     

      dep        jour  incid_hosp  incid_rea  incid_dc  incid_rad  \
26800  80  2020-03-19          29          7         1          3   
26801  80  2020-03-20          26          8         1          4   
26802  80  2020-03-21          30          6         5          5   
26803  80  2020-03-22          18          4         0          0   
26804  80  2020-03-23          14          2         1          1   
...    ..         ...         ...        ...       ...        ...   
27130  80  2021-02-12          19          3         3          9   
27131  80  2021-02-13           0          0         0          1   
27132  80  2021-02-14           0          0         0          0   
27133  80  2021-02-15          32          5         4          6   
27134  80  2021-02-16          16          4         6         11   

      departmentCode departmentName  regionCode       regionName  \
26800             80          Somme        32.0  Hauts-de-France   
26801             80          Somme

      dep        jour  incid_hosp  incid_rea  incid_dc  incid_rad  \
28140  84  2020-03-19           6          1         0          2   
28141  84  2020-03-20           2          0         0          1   
28142  84  2020-03-21           7          2         0          1   
28143  84  2020-03-22           3          0         0          1   
28144  84  2020-03-23           9          3         2          5   
...    ..         ...         ...        ...       ...        ...   
28470  84  2021-02-12          18          4         3         25   
28471  84  2021-02-13           8          1         0         10   
28472  84  2021-02-14           4          0         1          1   
28473  84  2021-02-15          30          2         9         15   
28474  84  2021-02-16          31          3         2         22   

      departmentCode departmentName  regionCode                  regionName  \
28140             84       Vaucluse        93.0  Provence-Alpes-Côte d'Azur   
28141        

      dep        jour  incid_hosp  incid_rea  incid_dc  incid_rad  \
29815  89  2020-03-19           2          2         2          0   
29816  89  2020-03-20           2          0         0          0   
29817  89  2020-03-21           2          0         0          0   
29818  89  2020-03-22           1          0         0          0   
29819  89  2020-03-23           3          4         0          0   
...    ..         ...         ...        ...       ...        ...   
30145  89  2021-02-12           5          1         3          9   
30146  89  2021-02-13           5          0         0          2   
30147  89  2021-02-14           4          2         1          2   
30148  89  2021-02-15           9          2         5          3   
30149  89  2021-02-16           8          1         2          9   

      departmentCode departmentName  regionCode               regionName  \
29815             89          Yonne        27.0  Bourgogne-Franche-Comté   
29816             8

      dep        jour  incid_hosp  incid_rea  incid_dc  incid_rad  \
31825  95  2020-03-19          80          8         4         15   
31826  95  2020-03-20          60         12         4         21   
31827  95  2020-03-21          74         10         4         18   
31828  95  2020-03-22          87         12         7          9   
31829  95  2020-03-23          72         15         9         13   
...    ..         ...         ...        ...       ...        ...   
32155  95  2021-02-12          28          0         6         22   
32156  95  2021-02-13          54          9         6         29   
32157  95  2021-02-14          11          2         1          4   
32158  95  2021-02-15           6          4         4         22   
32159  95  2021-02-16          26          3         1         28   

      departmentCode departmentName  regionCode     regionName  \
31825             95     Val-d'oise        11.0  Ile-de-France   
31826             95     Val-d'oise    